# BDD100K → YOLOv8 Data Processing
Chuyển đổi dataset BDD100K sang định dạng YOLOv8, chỉ giữ lại các phương tiện giao thông.

**Cấu trúc thư mục đầu vào:**
```
bdd100k/
├── bdd100k/
│   └── images/
│       ├── 100k/  (train, val, test)
│       └── 10k/   (train, val, test)
└── labels/
    ├── det_v2_train_release.json
    └── det_v2_val_release.json
```

**Output:** `yolo_dataset/` với cấu trúc chuẩn YOLOv8

## 1. Cài đặt thư viện

In [ ]:
# !pip install ultralytics tqdm Pillow opencv-python -q
!c:\Users\huynh\AppData\Local\Programs\Python\Python314\python.exe -m pip install ultralytics tqdm Pillow opencv-python -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import sys
print(sys.executable)

c:\Users\huynh\AppData\Local\Programs\Python\Python314\python.exe


In [14]:
import os, json, shutil, cv2
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import numpy as np

## 2. Mount Google Drive (nếu dữ liệu trên Drive)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# ========================
# ⚙️ CẤU HÌNH ĐƯỜNG DẪN
# Thay đổi theo vị trí thực tế của bạn
# ========================
# BDD100K_ROOT = 'D:/2026/DATN/Data/archive/bdd100k/bdd100k'  # Thư mục gốc bdd100k
# OUTPUT_DIR   = 'D:/2026/DATN/Data/preprocessing'            # Thư mục output
# IMAGE_SET    = '10k'  # '100k' hoặc '10k'

# # Đường dẫn tự động
# LABEL_TRAIN_JSON = f'{BDD100K_ROOT}/labels/det_v2_train_release.json'
# LABEL_VAL_JSON   = f'{BDD100K_ROOT}/labels/det_v2_val_release.json'
# IMG_TRAIN_DIR    = f'{BDD100K_ROOT}/bdd100k/images/{IMAGE_SET}/train'
# IMG_VAL_DIR      = f'{BDD100K_ROOT}/bdd100k/images/{IMAGE_SET}/val'

# ========================
# ⚙️ CẤU HÌNH ĐƯỜNG DẪN
# ========================
DATASET_ROOT = 'D:/2026/DATN/Data/archive'
OUTPUT_DIR   = 'D:/2026/DATN/Data/preprocessing'
IMAGE_SET    = '10k'

# Đường dẫn tự động
LABEL_TRAIN_JSON = f'{DATASET_ROOT}/labels/det_v2_train_release.json'
LABEL_VAL_JSON   = f'{DATASET_ROOT}/labels/det_v2_val_release.json'

# SỬA LẠI ĐƯỜNG DẪN ẢNH Ở ĐÂY
IMG_TRAIN_DIR    = f'{DATASET_ROOT}/bdd100k/images/{IMAGE_SET}/train'
IMG_VAL_DIR      = f'{DATASET_ROOT}/bdd100k/images/{IMAGE_SET}/val'

print(f'📁 Root     : {DATASET_ROOT}')
print(f'📁 Output   : {OUTPUT_DIR}')
print(f'🖼️  Image set: {IMAGE_SET}')
print(f'🏞️  Image Train Dir: {IMG_TRAIN_DIR}') # Thêm dòng này để kiểm tra

📁 Root     : D:/2026/DATN/Data/archive/bdd100k/bdd100k
📁 Output   : D:/2026/DATN/Data/preprocessing
🖼️  Image set: 10k


## 3. Cấu hình: Danh mục phương tiện giao thông

In [20]:
# ========================
# 5 lớp phương tiện giao thông
# ========================
TRAFFIC_CLASSES = [
    'car',      # 0
    'truck',    # 1
    'bus',      # 2
    'bicycle',  # 3
    'motor',    # 4 - xe máy
]

# Map category → class id (0-indexed)
CLASS_MAP = {name: idx for idx, name in enumerate(TRAFFIC_CLASSES)}

print('🏷️  Class mapping:')
for name, idx in CLASS_MAP.items():
    print(f'   {idx}: {name}')

🏷️  Class mapping:
   0: car
   1: truck
   2: bus
   3: bicycle
   4: motor


## 4. Hàm chuyển đổi box2d → YOLO format

In [21]:
def box2d_to_yolo(box2d, img_w, img_h):
    """
    Chuyển BDD100K box2d sang YOLO normalized format.
    
    BDD100K: {x1, y1, x2, y2} (pixel coords, top-left / bottom-right)
    YOLO:    x_center y_center width height (normalized 0..1)
    """
    x1 = float(box2d['x1'])
    y1 = float(box2d['y1'])
    x2 = float(box2d['x2'])
    y2 = float(box2d['y2'])

    # Clamp về trong ảnh
    x1 = max(0.0, min(x1, img_w))
    x2 = max(0.0, min(x2, img_w))
    y1 = max(0.0, min(y1, img_h))
    y2 = max(0.0, min(y2, img_h))

    w = x2 - x1
    h = y2 - y1

    # Bỏ bounding box có diện tích = 0
    if w <= 0 or h <= 0:
        return None

    x_center = (x1 + x2) / 2.0 / img_w
    y_center = (y1 + y2) / 2.0 / img_h
    w_norm   = w / img_w
    h_norm   = h / img_h

    return x_center, y_center, w_norm, h_norm


def process_split(json_path, img_src_dir, out_img_dir, out_lbl_dir,
                  class_map, img_w=1280, img_h=720,
                  copy_images=True, skip_no_label=True):
    """
    Xử lý một split (train / val).

    Tham số:
        json_path     : đường dẫn file JSON BDD100K
        img_src_dir   : thư mục ảnh gốc
        out_img_dir   : thư mục ảnh output
        out_lbl_dir   : thư mục label output (.txt)
        class_map     : dict {category: class_id}
        img_w/h       : kích thước ảnh mặc định (BDD100K = 1280×720)
        copy_images   : True = copy ảnh, False = chỉ tạo .txt
        skip_no_label : bỏ qua ảnh không có label hợp lệ sau khi lọc

    Trả về: (total, kept, skipped) thống kê
    """
    Path(out_img_dir).mkdir(parents=True, exist_ok=True)
    Path(out_lbl_dir).mkdir(parents=True, exist_ok=True)

    with open(json_path, 'r') as f:
        annotations = json.load(f)

    # BDD100K: root list hoặc dict với key 'frames' / 'labels'
    if isinstance(annotations, dict):
        annotations = annotations.get('frames', annotations.get('labels', []))

    total = len(annotations)
    kept = 0
    skipped = 0

    for entry in tqdm(annotations, desc=f'Processing {Path(json_path).stem}'):
        img_name = entry.get('name', '')
        labels   = entry.get('labels', []) or []

        # Thu thập các label hợp lệ
        yolo_lines = []
        for lbl in labels:
            cat   = lbl.get('category', '')
            box2d = lbl.get('box2d')

            # Lọc: chỉ giữ category trong class_map và có box2d
            if cat not in class_map or box2d is None:
                continue

            result = box2d_to_yolo(box2d, img_w, img_h)
            if result is None:
                continue

            cls_id = class_map[cat]
            xc, yc, w, h = result
            yolo_lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')

        # Bỏ qua ảnh không có object hợp lệ
        if skip_no_label and len(yolo_lines) == 0:
            skipped += 1
            continue

        stem = Path(img_name).stem  # tên file không đuôi

        # Ghi file label
        lbl_path = Path(out_lbl_dir) / f'{stem}.txt'
        with open(lbl_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

        # Copy ảnh
        if copy_images:
            src = Path(img_src_dir) / img_name
            dst = Path(out_img_dir) / img_name
            if src.exists():
                shutil.copy2(str(src), str(dst))
            else:
                print(f'⚠️  Ảnh không tìm thấy: {src}')

        kept += 1

    return total, kept, skipped

print('✅ Hàm xử lý đã được định nghĩa')

✅ Hàm xử lý đã được định nghĩa


## 5. Chạy xử lý dữ liệu

In [ ]:
# Tạo cấu trúc thư mục output chuẩn YOLOv8
OUT_TRAIN_IMG = f'{OUTPUT_DIR}/images/train'
OUT_TRAIN_LBL = f'{OUTPUT_DIR}/labels/train'
OUT_VAL_IMG   = f'{OUTPUT_DIR}/images/val'
OUT_VAL_LBL   = f'{OUTPUT_DIR}/labels/val'

# ──────────────────────────────
# Xử lý TRAIN split
# ──────────────────────────────
print('=' * 50)
print('📦 Xử lý TRAIN split...')
total_tr, kept_tr, skip_tr = process_split(
    json_path    = LABEL_TRAIN_JSON,
    img_src_dir  = IMG_TRAIN_DIR,
    out_img_dir  = OUT_TRAIN_IMG,
    out_lbl_dir  = OUT_TRAIN_LBL,
    class_map    = CLASS_MAP,
    copy_images  = True,
    skip_no_label= True,
)
print(f'Train  → tổng: {total_tr:,} | giữ lại: {kept_tr:,} | bỏ qua: {skip_tr:,}')

# ──────────────────────────────
# Xử lý VAL split
# ──────────────────────────────
print('=' * 50)
print('📦 Xử lý VAL split...')
total_vl, kept_vl, skip_vl = process_split(
    json_path    = LABEL_VAL_JSON,
    img_src_dir  = IMG_VAL_DIR,
    out_img_dir  = OUT_VAL_IMG,
    out_lbl_dir  = OUT_VAL_LBL,
    class_map    = CLASS_MAP,
    copy_images  = True,
    skip_no_label= True,
)
print(f'Val    → tổng: {total_vl:,} | giữ lại: {kept_vl:,} | bỏ qua: {skip_vl:,}')

print('=' * 50)
print(f'✅ Hoàn thành! Tổng ảnh: {kept_tr + kept_vl:,}')

## 6. Tạo file cấu hình dataset.yaml cho YOLOv8

In [ ]:
yaml_content = f"""# BDD100K - Traffic Objects Dataset
# Được tạo tự động từ script chuyển đổi BDD100K → YOLOv8

path: {OUTPUT_DIR}
train: images/train
val:   images/val

nc: {len(TRAFFIC_CLASSES)}  # số lượng lớp
names:
"""

for idx, name in enumerate(TRAFFIC_CLASSES):
    yaml_content += f'  {idx}: {name}\n'

yaml_path = f'{OUTPUT_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('📄 Nội dung dataset.yaml:')
print('-' * 40)
print(yaml_content)

## 7. Thống kê & kiểm tra chất lượng dataset

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def count_labels(label_dir, class_names):
    """Đếm số lượng instance của mỗi class trong thư mục label."""
    counter = Counter()
    label_files = list(Path(label_dir).glob('*.txt'))
    
    for lbl_file in label_files:
        with open(lbl_file, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    cls_id = int(line.split()[0])
                    counter[class_names[cls_id]] += 1
    return counter, len(label_files)

print('📊 THỐNG KÊ DATASET')
print('=' * 55)

train_cnt, n_train = count_labels(OUT_TRAIN_LBL, TRAFFIC_CLASSES)
val_cnt, n_val     = count_labels(OUT_VAL_LBL,   TRAFFIC_CLASSES)

print(f'{'Lớp':<15} {'Train':>10} {'Val':>10} {'Tổng':>10}')
print('-' * 55)
for cls in TRAFFIC_CLASSES:
    tr = train_cnt.get(cls, 0)
    vl = val_cnt.get(cls, 0)
    print(f'{cls:<15} {tr:>10,} {vl:>10,} {tr+vl:>10,}')
print('-' * 55)
print(f'{'TỔNG ẢNH':<15} {n_train:>10,} {n_val:>10,} {n_train+n_val:>10,}')

# Biểu đồ phân bố
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (cnt, title) in zip(axes, [(train_cnt, 'Train'), (val_cnt, 'Val')]):
    classes = TRAFFIC_CLASSES
    values  = [cnt.get(c, 0) for c in classes]
    colors  = ['#4C8BF5','#34A853','#EA4335','#FBBC05','#9B59B6','#1ABC9C','#E67E22']
    bars = ax.bar(classes, values, color=colors[:len(classes)], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{title} - Phân bố class', fontsize=13, pad=10)
    ax.set_xlabel('Category', fontsize=11)
    ax.set_ylabel('Số lượng instances', fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                f'{val:,}', ha='center', va='bottom', fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Biểu đồ đã lưu tại:', f'{OUTPUT_DIR}/class_distribution.png')

## 8. Kiểm tra trực quan bounding box

In [ ]:
import random
import matplotlib.patches as patches

COLORS = [
    '#FF3B3B','#FF9500','#FFCC00','#34C759',
    '#007AFF','#AF52DE','#FF2D55'
]

def visualize_sample(img_dir, lbl_dir, class_names, n=4, img_wh=(1280, 720)):
    """Hiển thị n ảnh mẫu kèm bounding box."""
    img_files = list(Path(img_dir).glob('*.jpg'))[:500]  # lấy từ 500 ảnh đầu
    samples   = random.sample(img_files, min(n, len(img_files)))

    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    axes = axes.flatten()

    for ax, img_path in zip(axes, samples):
        img  = Image.open(img_path).convert('RGB')
        w, h = img.size
        ax.imshow(img)

        lbl_path = Path(lbl_dir) / f'{img_path.stem}.txt'
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
                    x1 = (xc - bw/2) * w
                    y1 = (yc - bh/2) * h
                    color = COLORS[cls_id % len(COLORS)]
                    rect  = patches.Rectangle(
                        (x1, y1), bw*w, bh*h,
                        linewidth=2, edgecolor=color, facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.text(x1, y1-4, class_names[cls_id],
                            color='white', fontsize=9, fontweight='bold',
                            bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor='none'))

        ax.set_title(img_path.name, fontsize=9)
        ax.axis('off')

    plt.suptitle('Kiểm tra bounding box (mẫu ngẫu nhiên)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/sample_check.png', dpi=120, bbox_inches='tight')
    plt.show()

visualize_sample(OUT_TRAIN_IMG, OUT_TRAIN_LBL, TRAFFIC_CLASSES, n=4)
print('✅ Ảnh kiểm tra đã lưu tại:', f'{OUTPUT_DIR}/sample_check.png')

## 9. Huấn luyện YOLOv8

In [ ]:
from ultralytics import YOLO

# Chọn model: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large)
MODEL_SIZE = 'yolov8s'  # khuyến nghị: 's' cho cân bằng tốc độ/độ chính xác

model = YOLO(f'{MODEL_SIZE}.pt')  # tải pretrained weights

results = model.train(
    data    = yaml_path,
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,       # giảm xuống 8 nếu OOM
    workers = 4,
    device  = 0,        # GPU; dùng 'cpu' nếu không có GPU
    project = f'{OUTPUT_DIR}/runs',
    name    = 'bdd100k_traffic',
    exist_ok= True,
    patience= 10,       # early stopping
    save    = True,
    plots   = True,
)

print('🎉 Huấn luyện hoàn tất!')
print(f'📁 Kết quả tại: {OUTPUT_DIR}/runs/bdd100k_traffic/')

## 10. Đánh giá model trên tập val

In [ ]:
best_weights = f'{OUTPUT_DIR}/runs/bdd100k_traffic/weights/best.pt'
model_best   = YOLO(best_weights)

metrics = model_best.val(data=yaml_path, imgsz=640)

print('\n📈 KẾT QUẢ ĐÁNH GIÁ:')
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")

## 11. (Tùy chọn) Nén dataset để tải về

In [ ]:
import zipfile

# Chỉ nén labels và yaml (không nén ảnh để tiết kiệm dung lượng)
zip_path = '/content/yolo_labels_only.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['labels', 'dataset.yaml', 'class_distribution.png']:
        src = Path(OUTPUT_DIR) / folder
        if src.is_dir():
            for f in src.rglob('*'):
                zf.write(f, f.relative_to(OUTPUT_DIR))
        elif src.exists():
            zf.write(src, src.name)

print(f'✅ Đã nén: {zip_path}')
print(f'   Kích thước: {Path(zip_path).stat().st_size / 1024 / 1024:.1f} MB')

# Tải về
from google.colab import files
files.download(zip_path)